In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os
from pathlib import Path

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [2]:
# Paths
RAW_VIDEO_DIR = Path("../data/raw/video_data")
MODEL_PATH    = Path("../data/models/pose_landmarker_lite.task")
OUTPUT_CSV    = Path("../data/raw/tabular_data/angles_dataset.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Exercise folder names
EXERCISES = ["pullup", "pushup", "russian_twist", "squat"]

# Subfolders per exercise
CORRECTNESS_LABELS = ["correct", "incorrect"]

print("✅ Setup complete.")
print(f"   Model        : {MODEL_PATH.resolve()}")
print(f"   Video source : {RAW_VIDEO_DIR.resolve()}")
print(f"   Output CSV   : {OUTPUT_CSV.resolve()}")

✅ Setup complete.
   Model        : C:\Users\USER\Desktop\AI_PROJECT_2\data\models\pose_landmarker_lite.task
   Video source : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\video_data
   Output CSV   : C:\Users\USER\Desktop\AI_PROJECT_2\data\raw\tabular_data\angles_dataset.csv


In [3]:
def calculate_angle(a, b, c):
    """
    Calculate the angle at point B, formed by points A -> B -> C.
    
    Args:
        a, b, c: Each is a (x, y) tuple representing a joint coordinate.
    
    Returns:
        Angle in degrees (0–180).
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - \
              np.arctan2(a[1] - b[1], a[0] - b[0])
    
    angle = np.abs(np.degrees(radians))
    
    # Normalize to [0, 180]
    if angle > 180:
        angle = 360 - angle

    return round(angle, 2)


# Quick sanity check
test_angle = calculate_angle((0, 0), (1, 0), (1, 1))
print(f"✅ Angle function works. Test angle (expected 90°): {test_angle}°")

✅ Angle function works. Test angle (expected 90°): 90.0°


In [4]:
# MediaPipe Pose landmark indices (for reference)
LANDMARKS = {
    "left_shoulder"  : 11, "right_shoulder" : 12,
    "left_elbow"     : 13, "right_elbow"    : 14,
    "left_wrist"     : 15, "right_wrist"    : 16,
    "left_hip"       : 23, "right_hip"      : 24,
    "left_knee"      : 25, "right_knee"     : 26,
    "left_ankle"     : 27, "right_ankle"    : 28,
    "left_heel"      : 29, "right_heel"     : 30,
}

def get_coords(landmarks, name):
    """Extract (x, y) from a named landmark."""
    lm = landmarks[LANDMARKS[name]]
    return (lm.x, lm.y)

def get_z(landmarks, name):
    """Extract z (depth) from a named landmark."""
    return landmarks[LANDMARKS[name]].z


def extract_angles(landmarks):
    """
    Given a list of MediaPipe pose landmarks,
    compute and return the 10 joint angles + 3 torso rotation
    features as a dict.
    """
    angles = {}

    # Elbow angles: shoulder -> elbow -> wrist
    angles["left_elbow_angle"]  = calculate_angle(
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_elbow"),
        get_coords(landmarks, "left_wrist")
    )
    angles["right_elbow_angle"] = calculate_angle(
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_elbow"),
        get_coords(landmarks, "right_wrist")
    )

    # Shoulder angles: elbow -> shoulder -> hip
    angles["left_shoulder_angle"]  = calculate_angle(
        get_coords(landmarks, "left_elbow"),
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_hip")
    )
    angles["right_shoulder_angle"] = calculate_angle(
        get_coords(landmarks, "right_elbow"),
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_hip")
    )

    # Hip angles: shoulder -> hip -> knee
    angles["left_hip_angle"]  = calculate_angle(
        get_coords(landmarks, "left_shoulder"),
        get_coords(landmarks, "left_hip"),
        get_coords(landmarks, "left_knee")
    )
    angles["right_hip_angle"] = calculate_angle(
        get_coords(landmarks, "right_shoulder"),
        get_coords(landmarks, "right_hip"),
        get_coords(landmarks, "right_knee")
    )

    # Knee angles: hip -> knee -> ankle
    angles["left_knee_angle"]  = calculate_angle(
        get_coords(landmarks, "left_hip"),
        get_coords(landmarks, "left_knee"),
        get_coords(landmarks, "left_ankle")
    )
    angles["right_knee_angle"] = calculate_angle(
        get_coords(landmarks, "right_hip"),
        get_coords(landmarks, "right_knee"),
        get_coords(landmarks, "right_ankle")
    )

    # Ankle angles: knee -> ankle -> heel
    angles["left_ankle_angle"]  = calculate_angle(
        get_coords(landmarks, "left_knee"),
        get_coords(landmarks, "left_ankle"),
        get_coords(landmarks, "left_heel")
    )
    angles["right_ankle_angle"] = calculate_angle(
        get_coords(landmarks, "right_knee"),
        get_coords(landmarks, "right_ankle"),
        get_coords(landmarks, "right_heel")
    )

    # --- Torso rotation features (important for Russian twists) ---
    # z-depth difference between left and right landmarks:
    # positive = left side closer to camera, negative = right side closer
    shoulder_z_diff = round(get_z(landmarks, "left_shoulder") - get_z(landmarks, "right_shoulder"), 4)
    hip_z_diff      = round(get_z(landmarks, "left_hip")      - get_z(landmarks, "right_hip"),      4)

    # Torso rotation = how much shoulders have rotated RELATIVE to hips
    # Near 0 = no rotation (bad for Russian twist), large value = good rotation
    torso_rotation  = round(shoulder_z_diff - hip_z_diff, 4)

    angles["shoulder_z_diff"] = shoulder_z_diff
    angles["hip_z_diff"]      = hip_z_diff
    angles["torso_rotation"]  = torso_rotation

    return angles


print("✅ Landmark extraction function ready.")
print(f"   Joints tracked : {list(LANDMARKS.keys())}")
print(f"   Angles output  : 10 joint angles + 3 torso rotation features = 13 features per frame")

✅ Landmark extraction function ready.
   Joints tracked : ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'left_heel', 'right_heel']
   Angles output  : 10 joint angles + 3 torso rotation features = 13 features per frame


In [5]:
def process_video(video_path, exercise_name, exercise_correctness, options, video_id):
    """
    Process a single video file and return a list of rows,
    where each row represents one frame's angles + metadata.

    Args:
        video_path          : Path to the video file.
        exercise_name       : e.g. "squat"
        exercise_correctness: "correct" or "incorrect"
        options             : PoseLandmarkerOptions (initialized outside)

    Returns:
        List of dicts, one per valid frame.
    """
    rows = []
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"   ⚠️  Could not open: {video_path.name}")
        return rows

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30  # fallback

    prev_angles = None
    frame_number = 0

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to MediaPipe Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Timestamp in milliseconds (required for VIDEO mode)
            timestamp_ms = int((frame_number / fps) * 1000)

            # Run pose detection
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]  # first (and usually only) person
                angles    = extract_angles(landmarks)

                # --- Angular velocities (change in angle per second) ---
                if prev_angles is not None:
                    angular_velocities = {
                        f"{key}_velocity": round(abs(angles[key] - prev_angles[key]) * fps, 2)
                        for key in angles
                    }
                else:
                    # First valid frame: velocity is 0
                    angular_velocities = {f"{key}_velocity": 0.0 for key in angles}

                row = {
                    "video_id"            : video_id,
                    "frame_number"        : frame_number,
                    **angles,
                    **angular_velocities,
                    "exercise_name"       : exercise_name,
                    "exercise_correctness": exercise_correctness,
                }
                rows.append(row)
                prev_angles = angles

            frame_number += 1

    cap.release()
    return rows


print("✅ Video processing function ready.")
print("   Outputs per frame: frame_number + 10 angles + 10 angular velocities + 2 labels")

✅ Video processing function ready.
   Outputs per frame: frame_number + 10 angles + 10 angular velocities + 2 labels


In [6]:
def build_dataset(raw_video_dir, exercises, correctness_labels):
    """
    Walk through all exercise/correctness subfolders,
    process every video, and return the full combined DataFrame.
    """
    all_rows = []
    video_id_counter = 0

    # Initialize PoseLandmarker options once (shared across all videos)
    base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    options = PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=RunningMode.VIDEO,
        num_poses=1,                  # we only expect one person per video
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    for exercise in exercises:
        for correctness in correctness_labels:
            folder = raw_video_dir / exercise / correctness

            if not folder.exists():
                print(f"⚠️  Folder not found, skipping: {folder}")
                continue

            video_files = list(folder.glob("*.mp4")) + list(folder.glob("*.avi")) + list(folder.glob("*.mov"))

            if not video_files:
                print(f"⚠️  No videos found in: {folder}")
                continue

            print(f"\n📂 {exercise} / {correctness}  ({len(video_files)} videos)")

            for video_path in video_files:
                print(f"   🎞️  Processing: {video_path.name} ...", end=" ")
                rows = process_video(video_path, exercise, correctness, options, video_id_counter)
                all_rows.extend(rows)
                print(f"{len(rows)} frames extracted.")
                video_id_counter += 1

    df = pd.DataFrame(all_rows)
    return df


# --- Run it ---
print("🚀 Starting dataset creation...\n")
df_raw = build_dataset(RAW_VIDEO_DIR, EXERCISES, CORRECTNESS_LABELS)

print(f"\n✅ Done! Total frames collected: {len(df_raw)}")
print(f"   Shape : {df_raw.shape}")
print(f"   Columns: {list(df_raw.columns)}")

🚀 Starting dataset creation...


📂 pullup / correct  (11 videos)
   🎞️  Processing: pullups_1.mp4 ... 69 frames extracted.
   🎞️  Processing: pullups_10.mp4 ... 517 frames extracted.
   🎞️  Processing: pullups_11.mp4 ... 502 frames extracted.
   🎞️  Processing: pullups_2.mp4 ... 81 frames extracted.
   🎞️  Processing: pullups_3.mp4 ... 190 frames extracted.
   🎞️  Processing: pullups_4.mp4 ... 133 frames extracted.
   🎞️  Processing: pullups_5.mp4 ... 239 frames extracted.
   🎞️  Processing: pullups_6.mp4 ... 182 frames extracted.
   🎞️  Processing: pullups_7.mp4 ... 159 frames extracted.
   🎞️  Processing: pullups_8.mp4 ... 211 frames extracted.
   🎞️  Processing: pullups_9.mp4 ... 226 frames extracted.

📂 pullup / incorrect  (5 videos)
   🎞️  Processing: ipullups_1.mp4 ... 241 frames extracted.
   🎞️  Processing: ipullups_2.mp4 ... 699 frames extracted.
   🎞️  Processing: ipullups_3.mp4 ... 399 frames extracted.
   🎞️  Processing: ipullups_4.mp4 ... 173 frames extracted.
   🎞️  Proce

In [7]:
# --- Save to CSV ---
df_raw.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Raw dataset saved to: {OUTPUT_CSV}")
print(f"   Shape: {df_raw.shape}\n")

# --- Sanity Checks ---

# 1. Preview
print("=" * 60)
print("📋 First 5 rows:")
display(df_raw.head())

# 2. Class distribution
print("=" * 60)
print("📊 Exercise distribution:")
display(df_raw["exercise_name"].value_counts())

print("\n📊 Correctness distribution:")
display(df_raw["exercise_correctness"].value_counts())

print("\n📊 Exercise × Correctness breakdown:")
display(df_raw.groupby(["exercise_name", "exercise_correctness"]).size().unstack(fill_value=0))

# 3. Null check
print("=" * 60)
null_counts = df_raw.isnull().sum()
if null_counts.sum() == 0:
    print("✅ No null values found.")
else:
    print("⚠️  Null values detected:")
    display(null_counts[null_counts > 0])

# 4. Angle range check (all angles should be between 0 and 180)
print("=" * 60)
angle_cols = [col for col in df_raw.columns if "angle" in col and "velocity" not in col]
out_of_range = df_raw[angle_cols].apply(lambda col: ((col < 0) | (col > 180)).sum())

if out_of_range.sum() == 0:
    print("✅ All angles are within [0°, 180°].")
else:
    print("⚠️  Out-of-range angles detected:")
    display(out_of_range[out_of_range > 0])

# 5. Basic statistics
print("=" * 60)
print("📈 Angle columns statistics:")
display(df_raw[angle_cols].describe())

✅ Raw dataset saved to: ..\data\raw\tabular_data\angles_dataset.csv
   Shape: (19682, 30)

📋 First 5 rows:


,video_id,frame_number,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,...,right_hip_angle_velocity,left_knee_angle_velocity,right_knee_angle_velocity,left_ankle_angle_velocity,right_ankle_angle_velocity,shoulder_z_diff_velocity,hip_z_diff_velocity,torso_rotation_velocity,exercise_name,exercise_correctness
0,0,0,123.53,133.99,138.99,150.05,163.41,171.58,168.98,172.80,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,pullup,correct
1,0,1,118.37,131.03,136.90,149.61,164.01,171.91,166.61,173.30,...,9.9,71.1,15.0,153.3,146.1,3.18,2.29,0.89,pullup,correct
2,0,2,109.80,125.55,132.95,145.87,164.80,171.25,166.04,172.26,...,19.8,17.1,31.2,213.0,75.3,2.86,1.71,1.15,pullup,correct
3,0,3,104.63,121.92,129.91,143.21,165.99,171.22,166.99,172.69,...,0.9,28.5,12.9,106.8,1.2,0.87,0.56,0.31,pullup,correct
4,0,4,96.85,110.27,125.62,136.50,166.72,171.62,167.40,172.93,...,12.0,12.3,7.2,53.1,8.1,0.38,0.41,0.03,pullup,correct


📊 Exercise distribution:


exercise_name
russian_twist    6246
pullup           4610
pushup           4427
squat            4399
Name: count, dtype: int64


📊 Correctness distribution:


exercise_correctness
correct      10190
incorrect     9492
Name: count, dtype: int64


📊 Exercise × Correctness breakdown:


exercise_correctness,correct,incorrect
exercise_name,,
pullup,2509,2101
pushup,2295,2132
russian_twist,3116,3130
squat,2270,2129


✅ No null values found.
✅ All angles are within [0°, 180°].
📈 Angle columns statistics:


,left_elbow_angle,right_elbow_angle,left_shoulder_angle,right_shoulder_angle,left_hip_angle,right_hip_angle,left_knee_angle,right_knee_angle,left_ankle_angle,right_ankle_angle
count,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000,19682.000000
mean,107.358135,103.690163,65.923737,64.083454,119.738594,117.755323,132.686187,131.050652,139.774217,138.960039
std,49.910977,52.880983,49.364421,45.173673,47.871059,50.247072,36.936538,37.826333,20.077622,20.855371
min,0.090000,0.060000,0.000000,0.000000,0.040000,0.080000,0.190000,0.040000,0.030000,0.070000
25%,64.720000,58.060000,27.060000,29.135000,77.240000,73.332500,106.270000,101.622500,129.770000,128.502500
50%,114.805000,109.725000,53.020000,53.155000,133.655000,129.265000,139.470000,133.385000,141.185000,139.760000
75%,152.532500,154.137500,96.792500,96.547500,163.877500,165.660000,165.330000,165.660000,151.450000,151.720000
max,180.000000,180.000000,179.960000,179.990000,179.990000,180.000000,180.000000,180.000000,179.990000,179.990000
